In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler


In [2]:
df = pd.read_csv("D:/Course/python/energy forecasting/energy_env/data/processed/germany_hourly_processed.csv")
# Convert timestamp to datetime
df["time"] = pd.to_datetime(df["time"])

# Sort by time (VERY IMPORTANT)
df = df.sort_values("time").reset_index(drop=True)

print(df.head())
print(df.info())


                       time     load   solar     wind  wind_onshore  \
0 2015-01-01 07:00:00+00:00  41133.0    71.0  10208.0        9683.0   
1 2015-01-01 08:00:00+00:00  42963.0   773.0  10029.0        9502.0   
2 2015-01-01 09:00:00+00:00  45088.0  2117.0  10550.0       10025.0   
3 2015-01-01 10:00:00+00:00  47013.0  3364.0  11390.0       10862.0   
4 2015-01-01 11:00:00+00:00  48159.0  4198.0  12103.0       11575.0   

   wind_offshore  hour  day_of_week  day_of_month  month  is_weekend  \
0          525.0     7            3             1      1           0   
1          527.0     8            3             1      1           0   
2          525.0     9            3             1      1           0   
3          528.0    10            3             1      1           0   
4          528.0    11            3             1      1           0   

   is_holiday  load_forecast  
0           1        42522.0  
1           1        45020.0  
2           1        47101.0  
3           1   

In [3]:
import pandas as pd
import requests

url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": 52.52,        # Berlin (Germany proxy)
    "longitude": 13.405,
    "start_date": "2015-01-01",
    "end_date": "2024-01-01",
    "hourly": [
        "temperature_2m",
        "relative_humidity_2m",
        "wind_speed_10m",
        "precipitation",
        "weathercode"
    ],
    "timezone": "UTC"
}

response = requests.get(url, params=params)
data = response.json()

In [5]:
weather_df = pd.DataFrame({
    "time": pd.to_datetime(data["hourly"]["time"], utc=True),
    "temperature": data["hourly"]["temperature_2m"],
    "humidity": data["hourly"]["relative_humidity_2m"],
    "wind_speed": data["hourly"]["wind_speed_10m"],
    "precipitation": data["hourly"]["precipitation"],
    "weather_code": data["hourly"]["weathercode"]
})

print(weather_df.head())
print(weather_df.info())


                       time  temperature  humidity  wind_speed  precipitation  \
0 2015-01-01 00:00:00+00:00          3.8        96        14.4            0.0   
1 2015-01-01 01:00:00+00:00          3.6        95        14.9            0.0   
2 2015-01-01 02:00:00+00:00          3.3        94        14.6            0.0   
3 2015-01-01 03:00:00+00:00          3.0        94        14.1            0.0   
4 2015-01-01 04:00:00+00:00          2.3        94        13.0            0.0   

   weather_code  
0             2  
1             2  
2             3  
3             3  
4             2  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 78912 entries, 0 to 78911
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype              
---  ------         --------------  -----              
 0   time           78912 non-null  datetime64[ns, UTC]
 1   temperature    78912 non-null  float64            
 2   humidity       78912 non-null  int64              
 3   wind_speed  

In [6]:
df = df.merge(
    weather_df,
    on="time",
    how="left"
)

print(df[[
    "temperature",
    "humidity",
    "wind_speed",
    "precipitation",
    "weather_code"
]].isna().sum())


temperature      0
humidity         0
wind_speed       0
precipitation    0
weather_code     0
dtype: int64


In [12]:
WEATHER_COLS = [
    "temperature",
    "humidity",
    "wind_speed",
    "precipitation",
    "weather_code"
]


In [13]:
# # Set time as index (required for time interpolation)
df = df.set_index("time")

# # Interpolate ONLY weather columns
df[WEATHER_COLS] = df[WEATHER_COLS].interpolate(method="time")

# # Reset index back to column
df = df.reset_index()


In [23]:
print(df.columns)


Index(['time', 'load', 'solar', 'wind', 'wind_onshore', 'wind_offshore',
       'hour', 'day_of_week', 'day_of_month', 'month', 'is_weekend',
       'is_holiday', 'load_forecast', 'temperature', 'humidity', 'wind_speed',
       'precipitation', 'weather_code', 'temperature_lag_1', 'humidity_lag_1',
       'temperature_lag_24', 'humidity_lag_24', 'load_lag_1', 'load_lag_24',
       'load_lag_168', 'rolling_mean_24', 'rolling_std_24'],
      dtype='object')


In [17]:
df = df.sort_values("time").reset_index(drop=True)


In [19]:
LOAD_LAGS = [1, 24, 168]  # hour, day, week

for lag in LOAD_LAGS:
    df[f"load_lag_{lag}"] = df["load"].shift(lag)


In [20]:
df["rolling_mean_24"] = df["load"].rolling(window=24).mean()
df["rolling_std_24"]  = df["load"].rolling(window=24).std()


In [21]:
df = df.dropna().reset_index(drop=True)


In [22]:
print(df.filter(like="load_lag").columns)
print(df[[
    "load",
    "load_lag_1",
    "load_lag_24",
    "load_lag_168"
]].head())


Index(['load_lag_1', 'load_lag_24', 'load_lag_168'], dtype='object')
      load  load_lag_1  load_lag_24  load_lag_168
0  68569.0     65447.0      65964.0       41133.0
1  68599.0     68569.0      66400.0       42963.0
2  69484.0     68599.0      67746.0       45088.0
3  70635.0     69484.0      68507.0       47013.0
4  69962.0     70635.0      68100.0       48159.0


In [24]:
df.to_csv("final_energy_forecasting_dataset.csv", index=False)
